# Исследование: Юридический агент по ТК РФ

## Цель
Протестировать основные компоненты юридического агента:
- Подключение к Groq API (бесплатная LLM)
- Работа с векторным хранилищем (FAISS)
- RAG (Retrieval-Augmented Generation) для поиска по ТК РФ
- Промпты для юридических консультаций

## Стек
- **LLM**: Groq (llama-3.1-70b)
- **Embeddings**: sentence-transformers
- **Vector Store**: FAISS
- **Framework**: LangChain

## 1️⃣ Установка зависимостей

Запусти в терминале:
```bash
uv add groq langchain langchain-groq langchain-huggingface langchain-community langchain-text-splitters sentence-transformers faiss-cpu python-dotenv
```

In [ ]:
# Импорты
import os
import httpx
from dotenv import load_dotenv
from groq import Groq
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Загрузка переменных окружения
load_dotenv()

print("✅ Импорты загружены")

✅ Импорты загружены


## 2️⃣ Тест подключения к Groq API

**Получи бесплатный API ключ:**
1. Зайди на https://console.groq.com
2. Зарегистрируйся
3. Создай API ключ
4. Создай файл `.env` в корне проекта:
```
GROQ_API_KEY=your_api_key_here
```

In [12]:
# Проверка API ключа
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    print("❌ GROQ_API_KEY не найден в .env файле!")
else:
    print(f"✅ API ключ найден: {groq_api_key[:20]}...")

# Инициализация HTTP клиента с поддержкой SOCKS прокси
import httpx

# Получаем прокси из переменных окружения (если есть)
proxy = os.getenv("ALL_PROXY") or os.getenv("all_proxy")
if proxy:
    # Преобразуем socks:// в socks5://, который поддерживает httpx
    proxy = proxy.replace("socks://", "socks5://")
    http_client = httpx.Client(proxy=proxy)
    print(f"🌐 Используется прокси: {proxy}")
else:
    http_client = None
    print("🌐 Прокси не настроен")

# Инициализация клиента Groq
client = Groq(api_key=groq_api_key, http_client=http_client)

# Тестовый запрос (используем актуальную модель)
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # Обновленная модель
    messages=[
        {"role": "system", "content": "Ты — юридический консультант по Трудовому кодексу РФ."},
        {"role": "user", "content": "Что такое ТК РФ?"}
    ],
    temperature=0.3,
    max_tokens=500
)

print("\n🤖 Ответ модели:")
print(response.choices[0].message.content)

✅ API ключ найден: gsk_fWJB7YiO3sxZcwv1...
🌐 Используется прокси: socks5://127.0.0.1:12334/

🤖 Ответ модели:
ТК РФ - это Трудовой кодекс Российской Федерации. Это основной нормативный акт, регулирующий трудовые отношения в России. Трудовой кодекс устанавливает права, обязанности и гарантии работников и работодателей, а также регулирует различные аспекты трудовой деятельности, такие как заключение и расторжение трудовых договоров, режим рабочего времени, оплата труда, условия труда, социальное обеспечение и многое другое.

Трудовой кодекс РФ был принят 30 декабря 2001 года и вступил в силу 1 февраля 2002 года. С тех пор он неоднократно изменялся и дополнялся, чтобы отразить изменения в законодательстве и социально-экономических условиях страны.

ТК РФ состоит из 14 разделов, которые регулируют следующие вопросы:

* Общие положения
* Трудовой договор
* Режим рабочего времени
* Оплата труда
* Условия труда
* Социальное обеспечение
* Защита прав работников
* Решение индивидуальных трудовых

## 3️⃣ Создание тестовой базы знаний (ТК РФ)

Для прототипа используем несколько статей ТК РФ

In [13]:
# Тестовые статьи ТК РФ
tk_rf_articles = [
    """
    Статья 21. Основные права и обязанности работника
    Работник имеет право на:
    - заключение, изменение и расторжение трудового договора;
    - предоставление ему работы, обусловленной трудовым договором;
    - рабочее место, соответствующее условиям, предусмотренным государственными стандартами;
    - своевременную и в полном объеме выплату заработной платы;
    - отдых, обеспечиваемый установлением нормальной продолжительности рабочего времени;
    - полную достоверную информацию об условиях труда;
    - профессиональную подготовку, переподготовку и повышение своей квалификации.
    """,
    """
    Статья 80. Расторжение трудового договора по инициативе работника (по собственному желанию)
    Работник имеет право расторгнуть трудовой договор, предупредив об этом работодателя в письменной форме 
    не позднее чем за две недели, если иной срок не установлен настоящим Кодексом или иным федеральным законом.
    По соглашению между работником и работодателем трудовой договор может быть расторгнут и до истечения 
    срока предупреждения об увольнении.
    """,
    """
    Статья 114. Ежегодные оплачиваемые отпуска
    Работникам предоставляются ежегодные отпуска с сохранением места работы (должности) и среднего заработка.
    Оплачиваемый отпуск должен предоставляться работнику ежегодно.
    """,
    """
    Статья 115. Продолжительность ежегодного основного оплачиваемого отпуска
    Ежегодный основной оплачиваемый отпуск предоставляется работникам продолжительностью 28 календарных дней.
    """,
    """
    Статья 136. Порядок, место и сроки выплаты заработной платы
    Заработная плата выплачивается не реже чем каждые полмесяца. 
    Конкретная дата выплаты заработной платы устанавливается правилами внутреннего трудового распорядка, 
    коллективным договором или трудовым договором не позднее 15 календарных дней со дня окончания периода, 
    за который она начислена.
    """
]

# Создаем документы
documents = [Document(page_content=article) for article in tk_rf_articles]

print(f"✅ Создано {len(documents)} документов из ТК РФ")

✅ Создано 5 документов из ТК РФ


## 4️⃣ Создание векторного хранилища (FAISS)

Используем бесплатные embeddings от HuggingFace

In [14]:
# Инициализация embeddings (русскоязычная модель)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("✅ Embeddings модель загружена")

# Создание векторного хранилища
vector_store = FAISS.from_documents(documents, embeddings)

print("✅ Векторное хранилище создано")

✅ Embeddings модель загружена
✅ Векторное хранилище создано
✅ Векторное хранилище создано


## 5️⃣ Тест поиска по базе знаний

In [15]:
# Тестовый запрос
query = "Сколько дней отпуска положено работнику?"

# Поиск релевантных документов
relevant_docs = vector_store.similarity_search(query, k=2)

print(f"🔍 Запрос: {query}")
print(f"\n📄 Найдено документов: {len(relevant_docs)}\n")

for i, doc in enumerate(relevant_docs, 1):
    print(f"--- Документ {i} ---")
    print(doc.page_content.strip())
    print()

🔍 Запрос: Сколько дней отпуска положено работнику?

📄 Найдено документов: 2


📄 Найдено документов: 2

--- Документ 1 ---
--- Документ 1 ---
Статья 115. Продолжительность ежегодного основного оплачиваемого отпуска
    Ежегодный основной оплачиваемый отпуск предоставляется работникам продолжительностью 28 календарных дней.
Статья 115. Продолжительность ежегодного основного оплачиваемого отпуска
    Ежегодный основной оплачиваемый отпуск предоставляется работникам продолжительностью 28 календарных дней.


--- Документ 2 ---
Статья 114. Ежегодные оплачиваемые отпуска
    Работникам предоставляются ежегодные отпуска с сохранением места работы (должности) и среднего заработка.
    Оплачиваемый отпуск должен предоставляться работнику ежегодно.

--- Документ 2 ---
Статья 114. Ежегодные оплачиваемые отпуска
    Работникам предоставляются ежегодные отпуска с сохранением места работы (должности) и среднего заработка.
    Оплачиваемый отпуск должен предоставляться работнику ежегодно.



## 6️⃣ RAG: Ответ агента с использованием найденного контекста

In [18]:
# Настройка прокси для LangChain (если используется)
import os

# Сохраняем оригинальные прокси
original_all_proxy = os.environ.get("ALL_PROXY")
original_all_proxy_lower = os.environ.get("all_proxy")

# Конвертируем socks:// в socks5:// для всех прокси переменных
if original_all_proxy and original_all_proxy.startswith("socks://"):
    os.environ["ALL_PROXY"] = original_all_proxy.replace("socks://", "socks5://")
    print(f"🔧 Прокси ALL_PROXY: {os.environ['ALL_PROXY']}")

if original_all_proxy_lower and original_all_proxy_lower.startswith("socks://"):
    os.environ["all_proxy"] = original_all_proxy_lower.replace("socks://", "socks5://")
    print(f"🔧 Прокси all_proxy: {os.environ['all_proxy']}")

# Инициализация LangChain модели
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile",  # Обновленная модель
    temperature=0.3
)

# Восстанавливаем оригинальные прокси
if original_all_proxy:
    os.environ["ALL_PROXY"] = original_all_proxy
if original_all_proxy_lower:
    os.environ["all_proxy"] = original_all_proxy_lower

# Функция для получения ответа агента
def ask_legal_agent(question: str) -> str:
    # Поиск релевантных статей
    relevant_docs = vector_store.similarity_search(question, k=3)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    # Промпт для модели
    system_prompt = """
    Ты — юридический консультант-эксперт по Трудовому кодексу РФ.
    
    Твоя задача:
    1. Внимательно изучи предоставленные статьи ТК РФ
    2. Дай точный, юридически грамотный ответ на вопрос пользователя
    3. Обязательно укажи номера статей, на которые ссылаешься
    4. Отвечай понятным языком, но с юридической точностью
    5. Если в предоставленном контексте нет информации — честно скажи об этом
    
    Контекст (статьи ТК РФ):
    {context}
    """
    
    messages = [
        {"role": "system", "content": system_prompt.format(context=context)},
        {"role": "user", "content": question}
    ]
    
    response = llm.invoke(messages)
    return response.content

print("✅ Функция юридического агента готова")

🔧 Прокси all_proxy: socks5://127.0.0.1:12334/
✅ Функция юридического агента готова


## 7️⃣ Тестовые запросы

In [19]:
# Тест 1: Отпуск
question1 = "Сколько дней отпуска положено работнику по закону?"
print(f"❓ Вопрос: {question1}")
print(f"\n🤖 Ответ:\n{ask_legal_agent(question1)}")
print("\n" + "="*80 + "\n")

❓ Вопрос: Сколько дней отпуска положено работнику по закону?

🤖 Ответ:
Согласно статье 115 Трудового кодекса РФ, работнику положено 28 календарных дней ежегодного основного оплачиваемого отпуска. Это минимальная продолжительность отпуска, установленная законом.



🤖 Ответ:
Согласно статье 115 Трудового кодекса РФ, работнику положено 28 календарных дней ежегодного основного оплачиваемого отпуска. Это минимальная продолжительность отпуска, установленная законом.




In [20]:
# Тест 2: Увольнение
question2 = "За сколько дней нужно предупредить работодателя об увольнении?"
print(f"❓ Вопрос: {question2}")
print(f"\n🤖 Ответ:\n{ask_legal_agent(question2)}")
print("\n" + "="*80 + "\n")

❓ Вопрос: За сколько дней нужно предупредить работодателя об увольнении?

🤖 Ответ:
Согласно статье 80 Трудового кодекса РФ, работник должен предупредить работодателя об увольнении не позднее чем за две недели до даты увольнения, если иной срок не установлен федеральным законом или самим Кодексом. Однако по соглашению между работником и работодателем, трудовой договор может быть расторгнут и до истечения срока предупреждения об увольнении.



🤖 Ответ:
Согласно статье 80 Трудового кодекса РФ, работник должен предупредить работодателя об увольнении не позднее чем за две недели до даты увольнения, если иной срок не установлен федеральным законом или самим Кодексом. Однако по соглашению между работником и работодателем, трудовой договор может быть расторгнут и до истечения срока предупреждения об увольнении.




In [21]:
# Тест 3: Зарплата
question3 = "Как часто должны выплачивать зарплату?"
print(f"❓ Вопрос: {question3}")
print(f"\n🤖 Ответ:\n{ask_legal_agent(question3)}")
print("\n" + "="*80 + "\n")

❓ Вопрос: Как часто должны выплачивать зарплату?

🤖 Ответ:
Согласно статье 136 Трудового кодекса РФ, заработная плата должна выплачиваться не реже чем каждые полмесяца. Это означает, что работодатель обязан производить выплату зарплаты своим сотрудникам как минимум дважды в месяц. Конкретные даты выплаты могут быть установлены правилами внутреннего трудового распорядка, коллективным договором или трудовым договором, но не реже указанного выше срока.



🤖 Ответ:
Согласно статье 136 Трудового кодекса РФ, заработная плата должна выплачиваться не реже чем каждые полмесяца. Это означает, что работодатель обязан производить выплату зарплаты своим сотрудникам как минимум дважды в месяц. Конкретные даты выплаты могут быть установлены правилами внутреннего трудового распорядка, коллективным договором или трудовым договором, но не реже указанного выше срока.




In [22]:
# Тест 4: Права работника
question4 = "Какие основные права есть у работника?"
print(f"❓ Вопрос: {question4}")
print(f"\n🤖 Ответ:\n{ask_legal_agent(question4)}")

❓ Вопрос: Какие основные права есть у работника?

🤖 Ответ:
Согласно статье 21 Трудового кодекса РФ, работник имеет следующие основные права:

1. Заключение, изменение и расторжение трудового договора.
2. Право на предоставление ему работы, обусловленной трудовым договором.
3. Право на рабочее место, соответствующее условиям, предусмотренным государственными стандартами.
4. Право на своевременную и в полном объеме выплату заработной платы.
5. Право на отдых, обеспечиваемый установлением нормальной продолжительности рабочего времени.
6. Право на полную достоверную информацию об условиях труда.
7. Право на профессиональную подготовку, переподготовку и повышение своей квалификации.

Эти права гарантируются работнику и являются основой для его трудовой деятельности.

🤖 Ответ:
Согласно статье 21 Трудового кодекса РФ, работник имеет следующие основные права:

1. Заключение, изменение и расторжение трудового договора.
2. Право на предоставление ему работы, обусловленной трудовым договором.
3. 

## 8️⃣ Выводы

### Что работает:
- ✅ Подключение к Groq API (бесплатная LLM)
- ✅ Векторный поиск по ТК РФ (FAISS)
- ✅ RAG — модель использует найденный контекст
- ✅ Точные ответы со ссылками на статьи

### Следующие шаги:
1. Загрузить полный текст ТК РФ (397 статей)
2. Создать production код (`legal_agent.py`)
3. Добавить Telegram бота для удобного общения
4. Добавить историю диалога (memory)
5. Использовать LangGraph для сложных сценариев

### Стек технологий:
- **LLM**: Groq (llama-3.1-70b) — бесплатно
- **Embeddings**: sentence-transformers — бесплатно
- **Vector Store**: FAISS — бесплатно
- **Framework**: LangChain, LangGraph